# Jak dobre są modele prognozy energii

Notebook czyta wyniki z `wyniki/` (po `python main.py model`) i pokazuje:
1. ile model zyskuje względem prostych reguł,
2. gdzie się myli (doby suche vs mokre),
3. ile dają kolejne grupy danych (opad, pogoda, prognoza pogody),
4. jak prognoza wygląda na tle rzeczywistości – dobowo, godzinowo, miesięcznie,
5. z czego model korzysta (ważność cech) i czy przenosi się na nowe obiekty.

**WAPE** = suma |błędów| / suma zużycia [%] – im mniej, tym lepiej. Liczone na zbiorze testowym (ostatnie 30% czasu każdego obiektu), którego model nie widział.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

W = Path("wyniki")
TZ = "Europe/Warsaw"

GLOWNY = "xgb_prognoza_pogody"      # model, który pokazujemy (realny scenariusz: prognoza pogody z dnia poprzedniego)
BAZOWY = "profil_4tyg"              # najlepsza prosta reguła
GORNA = "xgb_pogoda_rzeczywista"    # górna granica: gdybyśmy znali pogodę idealnie
NAZWY = {GLOWNY: "model (prognoza pogody)", BAZOWY: "profil 4 tygodni", GORNA: "idealna pogoda",
         "naiwny_tydzien": "tydzień temu", "naiwny_doba": "wczoraj"}

met = pd.read_csv(W / "metryki.csv")
prog = pd.read_csv(W / "prognozy_test.csv.gz")
prog["czas"] = pd.to_datetime(prog["czas"], utc=True)
prog["czas_lok"] = prog["czas"].dt.tz_convert(TZ)
prog["dzien"] = prog["czas_lok"].dt.date
OBIEKTY = sorted(prog["obiekt_id"].unique())
print("warianty:", met["wariant"].nunique(), "| obiekty:", OBIEKTY)

## 1. Wynik w jednej tabeli
WAPE dobowe [%] na zbiorze testowym. Kolor: zielony = mało błędu, czerwony = dużo.

In [ ]:
tab = met.pivot(index="wariant", columns="obiekt", values="WAPE_doba")
tab = tab[[c for c in tab if c != "RAZEM"] + ["RAZEM"]].sort_values("RAZEM")
tab.round(1).style.background_gradient(cmap="RdYlGn_r", axis=None).format("{:.1f}", na_rep="–")

## 2. Model vs proste reguły
To najważniejszy wykres: o ile model jest lepszy od tego, co da się zrobić bez uczenia maszynowego.

In [ ]:
porown = [GLOWNY, BAZOWY, "naiwny_tydzien", GORNA]
t = met[met["wariant"].isin(porown) & (met["obiekt"] != "RAZEM")].pivot(index="obiekt", columns="wariant", values="WAPE_doba")[porown]
ax = t.rename(columns=NAZWY).plot.bar(figsize=(12, 5), width=0.8)
ax.set_ylabel("WAPE dobowe [%]"); ax.set_xlabel(""); ax.set_title("Błąd prognozy dobowej per obiekt (mniej = lepiej)")
ax.grid(axis="y", alpha=0.3); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

r = met[met["obiekt"] == "RAZEM"].set_index("wariant")["WAPE_doba"]
print(f"RAZEM: model {r[GLOWNY]:.1f}% | profil 4 tygodni {r[BAZOWY]:.1f}% | tydzień temu {r['naiwny_tydzien']:.1f}% | idealna pogoda {r[GORNA]:.1f}%")
print(f"Model ma o {100 * (1 - r[GLOWNY] / r[BAZOWY]):.0f}% mniejszy błąd niż profil 4 tygodni "
      f"i o {100 * (1 - r[GLOWNY] / r['naiwny_tydzien']):.0f}% mniejszy niż „tydzień temu”.")

## 3. Gdzie model się myli: doby suche vs mokre
Doba mokra = opad > 1 mm. Jeśli błąd siedzi w dobach mokrych, to ograniczeniem jest prognoza deszczu, a nie model.

In [ ]:
sm = met[met["wariant"].isin([GLOWNY, BAZOWY, GORNA])]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for a, kol, tytul in [(ax[0], "WAPE_doba_sucho", "doby suche"), (ax[1], "WAPE_doba_mokro", "doby mokre (opad > 1 mm)")]:
    t = sm.pivot(index="obiekt", columns="wariant", values=kol)[[GLOWNY, BAZOWY, GORNA]].rename(columns=NAZWY)
    t.plot.bar(ax=a, width=0.8, legend=a is ax[0]); a.set_title(tytul); a.set_xlabel(""); a.grid(axis="y", alpha=0.3)
    a.tick_params(axis="x", rotation=0)
ax[0].set_ylabel("WAPE dobowe [%]"); plt.tight_layout(); plt.show()
g = sm[sm["obiekt"] == "RAZEM"].set_index("wariant")
print(f"Model: doby suche {g.loc[GLOWNY, 'WAPE_doba_sucho']:.1f}% | doby mokre {g.loc[GLOWNY, 'WAPE_doba_mokro']:.1f}% "
      f"(idealna pogoda: {g.loc[GORNA, 'WAPE_doba_mokro']:.1f}%)")

## 4. Ile dają kolejne dane
Każdy krok dokłada grupę cech. Ostatni słupek (idealna pogoda) to górna granica, której nie da się osiągnąć w praktyce.

In [ ]:
kroki = [("profil_4tyg", "profil\n4 tygodni"), ("xgb_kalendarz_lagi", "kalendarz\n+ historia"),
         ("xgb_opad_przeszly", "+ opad\n(przeszły)"), ("xgb_pogoda_przeszla", "+ pogoda\n(przeszła)"),
         ("xgb_prognoza_pogody", "+ prognoza\npogody"), ("xgb_pogoda_rzeczywista", "idealna\npogoda")]
kroki = [(w, n) for w, n in kroki if w in r.index]
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ax[0].bar([n for _, n in kroki], [r[w] for w, _ in kroki], color=["#888780"] + ["#378ADD"] * (len(kroki) - 2) + ["#1D9E75"])
for i, (w, _) in enumerate(kroki):
    ax[0].text(i, r[w] + 0.3, f"{r[w]:.1f}", ha="center")
ax[0].set_title("RAZEM – WAPE dobowe [%]"); ax[0].grid(axis="y", alpha=0.3)
t = met[met["wariant"].isin([w for w, _ in kroki]) & (met["obiekt"] != "RAZEM")].pivot(index="wariant", columns="obiekt", values="WAPE_doba")
t.loc[[w for w, _ in kroki]].set_axis([n.replace("\n", " ") for _, n in kroki]).plot(ax=ax[1], marker="o")
ax[1].set_title("per obiekt"); ax[1].set_ylabel("WAPE dobowe [%]"); ax[1].grid(alpha=0.3); ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

## 5. Prognoza na tle rzeczywistości – dobowo
Tylko pełne doby (23–25 h). Zmień `OBIEKT`, żeby zobaczyć inny obiekt.

In [ ]:
OBIEKT = "RY_PZ1"      # <- ZMIEŃ: CH_T, RU_W, RU_ZMP, RY_PNB1, RY_PZ1

def dobowo(oid, kolumny):
    g = prog[prog["obiekt_id"] == oid]
    d = g.groupby("dzien")[["kwh", *kolumny]].sum(min_count=1)
    return d[g.groupby("dzien")["kwh"].count() >= 23]

d = dobowo(OBIEKT, [GLOWNY, BAZOWY])
fig, ax = plt.subplots(figsize=(15, 4.5))
ax.plot(d.index, d["kwh"], color="black", lw=1.6, label="rzeczywiste")
ax.plot(d.index, d[GLOWNY], lw=1.2, label=NAZWY[GLOWNY])
ax.plot(d.index, d[BAZOWY], lw=1, alpha=0.7, ls="--", label=NAZWY[BAZOWY])
ax.set_title(f"{OBIEKT} – zużycie energii [kWh / dobę], okres testowy"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Prognoza godzinowa – jeden tydzień
Czy model łapie rytm dobowy (noc / dzień) i weekendy.

In [ ]:
OD = None              # <- ZMIEŃ np. "2025-06-02"; None = pierwszy pełny tydzień testu
g = prog[prog["obiekt_id"] == OBIEKT].set_index("czas_lok").sort_index()
start = pd.Timestamp(OD, tz=TZ) if OD else g.index[0].normalize() + pd.Timedelta(days=1)
t = g.loc[start: start + pd.Timedelta(days=7)]
fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(t.index, t["kwh"], color="black", lw=1.5, label="rzeczywiste")
ax.plot(t.index, t[GLOWNY], lw=1.2, label=NAZWY[GLOWNY])
ax.set_title(f"{OBIEKT} – kWh / h, tydzień od {start:%Y-%m-%d}"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Prognoza vs rzeczywistość – wszystkie doby
Im bliżej przekątnej, tym lepiej. Punkty poniżej przekątnej = model zaniżył (zwykle dni z ulewą).

In [ ]:
fig, axes = plt.subplots(1, len(OBIEKTY), figsize=(4 * len(OBIEKTY), 4))
for a, oid in zip(np.atleast_1d(axes), OBIEKTY):
    d = dobowo(oid, [GLOWNY]).dropna()
    a.scatter(d["kwh"], d[GLOWNY], s=8, alpha=0.6)
    m = max(d["kwh"].max(), d[GLOWNY].max())
    a.plot([0, m], [0, m], color="gray", lw=1)
    a.set_title(oid); a.set_xlabel("rzeczywiste kWh/dobę"); a.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("prognoza kWh/dobę")
plt.tight_layout(); plt.show()

## 8. Rozkład błędów dobowych
Błąd dobowy w % zużycia. Wąski rozkład wokół zera = model trafia; długi lewy ogon = szczyty po ulewach, których nie przewidział.

In [ ]:
fig, axes = plt.subplots(1, len(OBIEKTY), figsize=(4 * len(OBIEKTY), 3.5), sharey=True)
for a, oid in zip(np.atleast_1d(axes), OBIEKTY):
    d = dobowo(oid, [GLOWNY]).dropna()
    blad = 100 * (d[GLOWNY] - d["kwh"]) / d["kwh"]
    a.hist(blad.clip(-60, 60), bins=40)
    a.axvline(0, color="black", lw=1)
    a.set_title(f"{oid}\nmediana |błędu| {blad.abs().median():.1f}%"); a.set_xlabel("błąd dobowy [%]")
plt.tight_layout(); plt.show()

## 9. Sumy miesięczne
Przy zakupie energii liczą się też sumy. Model celuje w medianę, więc na obiektach ze szczytami po ulewach zaniża sumy – tu widać o ile.

In [ ]:
p = prog.assign(miesiac=prog["czas_lok"].dt.tz_localize(None).dt.to_period("M"))
mies = p.groupby(["obiekt_id", "miesiac"])[["kwh", GLOWNY]].sum(min_count=1)
licz = p.groupby(["obiekt_id", "miesiac"])["kwh"].count()
mies = mies[licz >= 24 * 25]                        # tylko (prawie) pełne miesiące
mies["roznica_proc"] = 100 * (mies[GLOWNY] / mies["kwh"] - 1)
print(mies.round(1).to_string())
print("\nśrednia różnica miesięczna per obiekt [%]:")
print(mies.groupby("obiekt_id")["roznica_proc"].mean().round(1).to_string())

## 10. Z czego model korzysta
Ważność cech w modelu głównym (XGBoost). Grupy: kalendarz, historia zużycia, opad, pogoda, prognoza pogody.

In [ ]:
waz = pd.read_csv(W / f"waznosc_{GLOWNY}.csv", index_col=0)["waznosc"]
def grupa(c):
    if c.startswith("prog_"): return "prognoza pogody"
    if c.startswith("opad"): return "opad (przeszły)"
    if c.startswith(("kwh_", "profil", "poziom")): return "historia zużycia"
    if c.startswith("ob_"): return "obiekt"
    if c.endswith("_pr") or c.endswith("_rz"): return "pogoda (przeszła)"
    return "kalendarz"
kolory = {"prognoza pogody": "#1D9E75", "opad (przeszły)": "#378ADD", "historia zużycia": "#7F77DD",
          "obiekt": "#888780", "pogoda (przeszła)": "#85B7EB", "kalendarz": "#EF9F27"}
top = waz.head(15)[::-1]
fig, ax = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2, 1]})
ax[0].barh(top.index, top.values, color=[kolory[grupa(c)] for c in top.index]); ax[0].set_title("15 najważniejszych cech")
g = waz.groupby(waz.index.map(grupa)).sum().sort_values()
ax[1].barh(g.index, g.values, color=[kolory[i] for i in g.index]); ax[1].set_title("ważność wg grupy")
plt.tight_layout(); plt.show()

## 11. Czy model przenosi się na nowe obiekty (Rybnik)
`xgb_transfer_RY` uczony był **bez danych z Rybnika** (tylko Chorzów i Ruda). Jeśli wynik jest bliski modelowi, który Rybnik widział, model zadziała też na innych pompowniach.

In [ ]:
ry = met[met["obiekt"].isin(["RY_PNB1", "RY_PZ1"]) & met["wariant"].isin([GLOWNY, "xgb_transfer_RY", "xgb_transfer_RY_prognoza", BAZOWY])]
t = ry.pivot(index="obiekt", columns="wariant", values="WAPE_doba")
t = t[[c for c in [GLOWNY, "xgb_transfer_RY", "xgb_transfer_RY_prognoza", BAZOWY] if c in t]]
ax = t.rename(columns={**NAZWY, "xgb_transfer_RY": "bez Rybnika w treningu", "xgb_transfer_RY_prognoza": "bez Rybnika + prognoza"}).plot.bar(figsize=(10, 4.5), width=0.8)
ax.set_ylabel("WAPE dobowe [%]"); ax.set_xlabel(""); ax.grid(axis="y", alpha=0.3); plt.xticks(rotation=0); plt.tight_layout(); plt.show()
print(t.round(1).to_string())

## 12. Podsumowanie liczbowe (do slajdu)

In [ ]:
g = met[met["obiekt"] == "RAZEM"].set_index("wariant")
po = met[(met["wariant"] == GLOWNY) & (met["obiekt"] != "RAZEM")].set_index("obiekt")
print(f"• Model ({GLOWNY}): WAPE dobowe {g.loc[GLOWNY, 'WAPE_doba']:.1f}% "
      f"vs profil 4 tygodni {g.loc[BAZOWY, 'WAPE_doba']:.1f}% → {100 * (1 - g.loc[GLOWNY, 'WAPE_doba'] / g.loc[BAZOWY, 'WAPE_doba']):.0f}% mniej błędu")
print(f"• Doby suche: {g.loc[GLOWNY, 'WAPE_doba_sucho']:.1f}% | doby mokre: {g.loc[GLOWNY, 'WAPE_doba_mokro']:.1f}%")
print(f"• Z idealną pogodą: {g.loc[GORNA, 'WAPE_doba']:.1f}% – tyle dałaby lepsza prognoza deszczu")
print("• Per obiekt: " + ", ".join(f"{o} {v:.1f}%" for o, v in po["WAPE_doba"].items()))
if "xgb_transfer_RY" in g.index:
    tr = met[(met["wariant"] == "xgb_transfer_RY") & met["obiekt"].str.startswith("RY")].set_index("obiekt")["WAPE_doba"]
    print("• Bez Rybnika w treningu: " + ", ".join(f"{o} {v:.1f}%" for o, v in tr.items()))